# Session 32: Logistic Regression Classification Baseline
**Week 3 Baseline Classifier for At-Risk Target Prediction**

In this notebook, we shift from regression tasks to binary classification. We construct an "at-risk" target variable ($G3 < 10$) and fit a scaled Logistic Regression baseline model using a scikit-learn `Pipeline`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, classification_report, confusion_matrix
)

# Resolve project directories
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / ".git").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / ".git").exists():
            PROJECT_ROOT = parent
            break

DATA_DIRECTORY = PROJECT_ROOT / "data"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"
TABLES_DIRECTORY = REPORTS_DIRECTORY / "tables"

TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("Project Root:", PROJECT_ROOT)

Project Root: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml


In [2]:
def load_and_transform_data():
    # Attempt to load the same feature splits used in regression
    # If loading raw processed table, split using the same random state
    from sklearn.model_selection import train_test_split
    
    candidates = list((DATA_DIRECTORY / "processed").rglob("*.parquet")) + list((DATA_DIRECTORY / "processed").rglob("*.csv"))
    for path in candidates:
        if any(term in path.name.lower() for term in ["comparison", "prediction", "result"]):
            continue
        try:
            table = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
            if "G3" in table.columns:
                X = table.drop(columns=["G3"]).copy()
                X = pd.get_dummies(X, drop_first=True, dtype=float)
                y_reg = table["G3"]
                
                # Critical step: Convert regression target to binary classification target
                # At-risk (1) if G3 < 10, otherwise Pass (0)
                y_clf = (y_reg < 10).astype(int)
                
                Xtr, Xte, yctr, ycte = train_test_split(
                    X, y_clf, test_size=0.20, random_state=42
                )
                return Xtr, Xte, yctr, ycte
        except Exception as e:
            continue
    raise FileNotFoundError("Could not locate source data splits.")

Xtr_f, Xte_f, yctr, ycte = load_and_transform_data()
print(f"Train shapes: X={Xtr_f.shape}, y={yctr.shape}")
print(f"At-risk prevalence in training: {yctr.mean():.2%}")

Train shapes: X=(316, 41), y=(316,)
At-risk prevalence in training: 32.59%


In [3]:
# Instantiate scaled Logistic Regression pipeline
clf = make_pipeline(
    StandardScaler(), 
    LogisticRegression(max_iter=1000, random_state=42)
)

# Fit classifier on binarized training data
clf.fit(Xtr_f, yctr)

# Generate class and probability predictions
predictions = clf.predict(Xte_f)
probabilities = clf.predict_proba(Xte_f)[:, 1]

# Compute standard evaluation metrics
accuracy = accuracy_score(ycte, predictions)
precision = precision_score(ycte, predictions, zero_division=0)
recall = recall_score(ycte, predictions, zero_division=0)
f1 = f1_score(ycte, predictions, zero_division=0)
roc_auc = roc_auc_score(ycte, probabilities)

print("Baseline Logistic Regression Performance:")
print("-" * 40)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}\n")

print("Classification Report:")
print(classification_report(ycte, predictions))

Baseline Logistic Regression Performance:
----------------------------------------
Accuracy:  0.8987
Precision: 0.8276
Recall:    0.8889
F1 Score:  0.8571
ROC AUC:   0.9736

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92        52
           1       0.83      0.89      0.86        27

    accuracy                           0.90        79
   macro avg       0.88      0.90      0.89        79
weighted avg       0.90      0.90      0.90        79



In [4]:
classification_path = TABLES_DIRECTORY / "classification_model_comparison.csv"

# Construct the output tracking row matching target definitions
log_row = pd.DataFrame([{
    "Session": 32,
    "Week": 3,
    "Task Type": "Classification",
    "Scenario": "Full-information",
    "Feature Set": "X_full",
    "Target": "At-Risk (G3 < 10)",
    "Model": "Logistic Regression",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "ROC AUC": roc_auc
}])

if classification_path.exists():
    comparison_table = pd.read_csv(classification_path)
    comparison_table = comparison_table[comparison_table["Session"] != 32].copy()
    comparison_table = pd.concat([comparison_table, log_row], ignore_index=True)
else:
    comparison_table = log_row.copy()

comparison_table.to_csv(classification_path, index=False)
print("Updated classification comparison table saved at:")
print(classification_path)
display(comparison_table.round(4))

Updated classification comparison table saved at:
/home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml/reports/tables/classification_model_comparison.csv


,Session,Week,Task Type,Scenario,Feature Set,Target,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,32,3,Classification,Full-information,X_full,At-Risk (G3 < 10),Logistic Regression,0.8987,0.8276,0.8889,0.8571,0.9736


In [5]:
# Validate scaling step and architecture parameters
assert isinstance(clf.steps[0][1], StandardScaler), "Pipeline step 0 must be StandardScaler!"
assert isinstance(clf.steps[1][1], LogisticRegression), "Pipeline step 1 must be LogisticRegression!"
assert clf.steps[1][1].max_iter == 1000, "max_iter must be set to 1000!"

# Verify data shapes and types
assert set(np.unique(yctr)).issubset({0, 1}), "Target classes must be strictly binary 0 and 1!"
assert len(predictions) == len(ycte), "Prediction size mismatch!"

print("SESSION 32 NOTEBOOK SECTION COMPLETED SUCCESSFULLY")

SESSION 32 NOTEBOOK SECTION COMPLETED SUCCESSFULLY
